<div dir="rtl" align="right">

# بیز ساده (Naive Bayes)

درسی از مدرسه پایتون و هوش مصنوعی، دکتر بهروز طاهری

## ایده در یک نگاه
بیز ساده از **قضیه بیز** استفاده می‌کند تا احتمال هر کلاس را با دیدن ویژگی‌ها حساب کند:

$$P(c\mid x)=\frac{P(x\mid c)\,P(c)}{P(x)}$$

- $P(c)$: احتمال **پیشین**؛ قبل از دیدن داده (مثلاً ۳۰٪ ایمیل‌ها اسپم‌اند).
- $P(x\mid c)$: **درست‌نمایی**؛ این ویژگی‌ها در کلاس $c$ چقدر رایج‌اند.
- $P(c\mid x)$: احتمال **پسین**؛ بعد از دیدن داده.

## فرض «ساده»
محاسبه $P(x_1,\dots,x_d\mid c)$ سخت است. فرض می‌کنیم ویژگی‌ها **با دانستن کلاس، مستقل‌اند**:

$$P(c\mid x)\propto P(c)\prod_{j=1}^{d}P(x_j\mid c)$$

این فرض معمولاً درست نیست («رایگان» و «جایزه» در اسپم با هم می‌آیند)، ولی مدل در عمل خیلی خوب کار می‌کند، چون برای تصمیم فقط **ترتیب** احتمال‌ها مهم است.

## مثال کامل: فیلتر اسپم
از ۱۰۰ ایمیل، ۳۰ تا اسپم. ایمیل جدید شامل «رایگان» و «جلسه» است:

| کلمه | P(کلمه ∣ اسپم) | P(کلمه ∣ عادی) |
|---|---|---|
| رایگان | 0.60 | 0.05 |
| جلسه | 0.10 | 0.40 |

- اسپم: $0.3\times0.6\times0.1=0.018$
- عادی: $0.7\times0.05\times0.4=0.014$

نرمال‌سازی: $P(\text{spam}\mid x)=0.018/(0.018+0.014)=0.56$ ← **اسپم**.



</div>

In [ ]:
# The spam example, step by step
prior = {"spam": 0.3, "normal": 0.7}
like = {"spam": {"free": 0.60, "meeting": 0.10}, "normal": {"free": 0.05, "meeting": 0.40}}
email = ["free", "meeting"]

score = {}
for c in prior:
    s = prior[c]
    for w in email:
        s *= like[c][w]
    score[c] = s
total = sum(score.values())
for c in score:
    print(c, "score =", round(score[c], 4), " posterior =", round(score[c] / total, 3))

<div dir="rtl" align="right">



## سه نوع رایج
| نوع | ویژگی‌ها | $P(x_j\mid c)$ | کاربرد |
|---|---|---|---|
| `GaussianNB` | پیوسته | توزیع نرمال با میانگین و واریانس هر کلاس | داده عددی |
| `MultinomialNB` | شمارش | فراوانی کلمه | دسته‌بندی متن |
| `BernoulliNB` | ۰/۱ | بود یا نبود کلمه | متن کوتاه |

> [بیز گاوسی در یک بعد: چگالی هر کلاس ضرب در پیشین، و احتمال پسین که از ۰٫۵ می‌گذرد.](https://raw.githubusercontent.com/behrooztaheri/learn-python-ai/main/assets/img/lessons/nb_gauss.png)

*بیز گاوسی در یک بعد: چگالی هر کلاس ضرب در پیشین، و احتمال پسین که از ۰٫۵ می‌گذرد.*

## هموارسازی لاپلاس
اگر کلمه‌ای در آموزش هرگز در اسپم نیامده باشد، $P=0$ و کل حاصل‌ضرب صفر می‌شود. با `alpha=1` به همه شمارش‌ها ۱ اضافه می‌کنیم:

$$P(w\mid c)=\frac{\text{count}(w,c)+\alpha}{\text{count}(c)+\alpha V}$$

[[lab:cls_nb]]



</div>

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline

texts = ["win a free prize now", "free money click now", "meeting at ten tomorrow", "project meeting notes",
         "claim your free gift", "lunch with the team tomorrow", "cheap prize offer now", "notes from the team meeting"]
labels = ["spam", "spam", "normal", "normal", "spam", "normal", "spam", "normal"]

model = make_pipeline(CountVectorizer(), MultinomialNB(alpha=1.0)).fit(texts, labels)
for t in ["free prize meeting", "team lunch tomorrow", "click now for money"]:
    p = model.predict_proba([t])[0]
    print(f"{t!r:25} -> {model.predict([t])[0]:7s}", dict(zip(model.classes_, p.round(3))))

<div dir="rtl" align="right">



| مزایا | معایب |
|---|---|
| فوق‌العاده سریع، حتی با میلیون‌ها ویژگی | فرض استقلال |
| با داده کم کار می‌کند | احتمال‌ها معمولاً خیلی نزدیک ۰ یا ۱ (بدکالیبره) |
| پایه عالی برای متن | مرزهای ساده |

</div>

<div dir="rtl" align="right">

### تمرین

GaussianNB را روی داده سرطان سینه با ۵ تا ارزیابی کن و با LogisticRegression مقایسه کن.

</div>

In [ ]:
# Your answer here